# 05 — Analyse détaillée des meilleurs modèles

But : comprendre les erreurs avant de faire le modèle final. On sauvegarde :

- classification report par modèle ;
- matrice de confusion ;
- fichiers mal classés ;
- résumé des métriques.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = c:\Users\yeyuy\Downloads\projet_ml_audio_classique_v7_structured\projet_ml_audio_classique_v7_structured


In [2]:
import pandas as pd
from src.config import FEATURE_DIR, RESULT_DIR
from src.pipeline_steps import step_error_analysis
features = pd.read_parquet(FEATURE_DIR / 'train_audio_features.parquet')
cv_results = pd.read_csv(RESULT_DIR / 'model_comparison_cv.csv')
TOP_MODELS = cv_results.sort_values('macro_f1_mean', ascending=False).head(3)['model'].tolist()
print(TOP_MODELS)

['hist_gradient_boosting', 'extra_trees', 'cosine_knn']


In [3]:
analysis = step_error_analysis(features, TOP_MODELS, n_splits=5)
display(analysis)

,macro_f1,weighted_f1,balanced_accuracy,accuracy,model
0,0.739969,0.768080,0.694628,0.768631,hist_gradient_boosting
1,0.690825,0.724447,0.780560,0.716658,extra_trees
2,0.661457,0.653316,0.638237,0.653291,cosine_knn


## Lire les erreurs

Si les erreurs concernent surtout des classes rares, le problème est le manque de données. Si elles concernent des classes fréquentes, il faut améliorer preprocessing/features ou choisir un modèle non linéaire.

In [4]:
for model in TOP_MODELS:
    path = RESULT_DIR / f'errors_{model}.csv'
    if path.exists():
        err = pd.read_csv(path)
        print('', model, 'nb erreurs:', len(err))
        display(err[['filename','true_label','pred_label']].head(10))

 hist_gradient_boosting nb erreurs: 24511


,filename,true_label,pred_label
0,iNat1114648.ogg,1161364,brnowl
1,iNat1114648.ogg,1161364,whtdov
2,iNat1114648.ogg,1161364,brnowl
3,iNat1264238.ogg,1161364,shtnig1
4,iNat1264238.ogg,1161364,shtnig1
5,iNat1264238.ogg,1161364,shtnig1
6,iNat818781.ogg,1161364,24279
7,iNat840159.ogg,1161364,sobtyr1
8,iNat840159.ogg,1161364,socfly1
9,iNat840159.ogg,1161364,sobtyr1


 extra_trees nb erreurs: 30017


,filename,true_label,pred_label
0,iNat1264238.ogg,1161364,shtnig1
1,iNat1264238.ogg,1161364,shtnig1
2,iNat1264238.ogg,1161364,shtnig1
3,iNat840159.ogg,1161364,23176
4,iNat840159.ogg,1161364,23176
5,iNat1460166.ogg,116570,horscr1
6,iNat1269019.ogg,1176823,greela
7,iNat145706.ogg,1176823,22973
8,iNat145706.ogg,1176823,smbtin1
9,iNat145706.ogg,1176823,22973


 cosine_knn nb erreurs: 36730


,filename,true_label,pred_label
0,iNat1264238.ogg,1161364,grasal3
1,iNat1264238.ogg,1161364,orwpar
2,iNat1264238.ogg,1161364,grasal3
3,iNat556514.ogg,1161364,coffal1
4,iNat810195.ogg,1161364,whtdov
5,iNat818781.ogg,1161364,rufnig1
6,iNat818781.ogg,1161364,rufnig1
7,iNat818781.ogg,1161364,rufnig1
8,iNat840159.ogg,1161364,eulfly1
9,iNat840159.ogg,1161364,platyr1


Les erreurs observées concernent principalement des espèces partageant des signatures fréquentielles similaires ou des enregistrements bruités. Plusieurs fichiers sont également mal classés de manière répétée selon les segments extraits, ce qui suggère une forte variabilité intra-classe et des frontières de décision parfois ambiguës.

Le grand nombre d'erreurs sur certaines espèces indique aussi un manque de représentativité pour plusieurs classes rares. À l'inverse, certaines confusions persistent sur des classes fréquentes, montrant que les limitations proviennent également des caractéristiques extraites et du décalage entre train_audio et soundscapes.


-> on choisit modèle non linéaire pour la phase finale ainsi qu'une stratégie de fusion de scores et de prédictions par segments afin de réduire les erreurs locales sur les longues séquences audio.